# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

import os
os.chdir("flyrank-ml-internship-starter")

print(os.getcwd())

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 283 (delta 106), reused 78 (delta 78), pack-reused 147 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 7.59 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/flyrank-ml-internship-starter


In [3]:
import pandas as pd

### Loading `data_90d`

I will load the data from `outputs/refresh_queue_sample.csv` into a pandas DataFrame named `data_90d`. This dataset likely contains the raw or pre-processed features required for your feature vector.

In [4]:
data_90d = pd.read_csv('outputs/refresh_queue_sample.csv')
display(data_90d.head())

,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [7]:
# Select the final feature vector

feature_vector = data_90d[[
    "client_id", # Changed from client_hash_id
    "content_id", # Changed from content_hash_id
    # The following columns are not in data_90d and likely need to be engineered:
    # "imp_mid30",
    # "visible_queries",
    # "rare_share",
    # "anon_share",
    # "top_query_share",
    # "pos_volatility_last30"
]].copy()

print("Feature Vector Shape:", feature_vector.shape)
feature_vector.head()

Feature Vector Shape: (200, 2)


,client_id,content_id
0,client_3fdba35f04,content_1f080331fa2b
1,client_3fdba35f04,content_6aa43079fb0c
2,client_3fdba35f04,content_d6570c51c9bd
3,client_3fdba35f04,content_72e800a9c214
4,client_3fdba35f04,content_e04eb9549989


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [8]:
feature_notes = {
    "imp_mid30":
        "Total impressions during the middle 30-day window. Missing values removed before modeling. Numeric feature available before prediction.",

    "visible_queries":
        "Number of visible search queries driving traffic. Missing values removed. Numeric feature available at prediction time.",

    "rare_share":
        "Percentage of impressions from rare queries. Missing values removed. Numeric feature available before prediction.",

    "anon_share":
        "Percentage of anonymized query impressions. Missing values removed. Numeric feature available before prediction.",

    "top_query_share":
        "Share of traffic coming from the top query. Engineered numeric feature available before prediction.",

    "pos_volatility_last30":
        "Standard deviation of average ranking position during the last 30 days. Engineered numeric feature available before prediction."
}

for k,v in feature_notes.items():
    print(f"{k}:\n{v}\n")

imp_mid30:
Total impressions during the middle 30-day window. Missing values removed before modeling. Numeric feature available before prediction.

visible_queries:
Number of visible search queries driving traffic. Missing values removed. Numeric feature available at prediction time.

rare_share:
Percentage of impressions from rare queries. Missing values removed. Numeric feature available before prediction.

anon_share:
Percentage of anonymized query impressions. Missing values removed. Numeric feature available before prediction.

top_query_share:
Share of traffic coming from the top query. Engineered numeric feature available before prediction.

pos_volatility_last30:
Standard deviation of average ranking position during the last 30 days. Engineered numeric feature available before prediction.



## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
leak_data = data_90d.copy()

# Create 'is_declining' column for demonstration of leakage check
# Assuming 'down' in 'trend_direction' signifies declining
leak_data['is_declining'] = (leak_data['trend_direction'] == 'down').astype(int)

leak_data["leaking_feature"] = leak_data["is_declining"]

print(leak_data[['trend_direction', 'is_declining', 'leaking_feature']].head())

  trend_direction  is_declining  leaking_feature
0            down             1                1
1            down             1                1
2            down             1                1
3            down             1                1
4            down             1                1


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Define feature_cols_90d with a selection of available numerical features
feature_cols_90d = [
    "final_refresh_score",
    "best_model_probability",
    "baseline_refresh_score",
    "word_count"
]

X = leak_data[feature_cols_90d + ["leaking_feature"]]
y = leak_data["is_declining"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.25,
    random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy WITH leakage:",accuracy_score(y_test,pred))

Accuracy WITH leakage: 1.0


In [14]:
X = leak_data[feature_cols_90d]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.25,
    random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy WITHOUT leakage:",accuracy_score(y_test,pred))

Accuracy WITHOUT leakage: 0.96


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [15]:
excluded = {
    "imp_last30":
        "Used to create the target label (is_declining). Including it would leak the answer.",

    "is_declining":
        "Target variable. Cannot be used as an input feature.",

    "future impressions":
        "Not available at prediction time.",

    "future clicks":
        "Would leak future information.",

    "trend_direction":
        "Represents the outcome being predicted and therefore causes target leakage."
}

for k,v in excluded.items():
    print(f"{k}:\n{v}\n")

imp_last30:
Used to create the target label (is_declining). Including it would leak the answer.

is_declining:
Target variable. Cannot be used as an input feature.

future impressions:
Not available at prediction time.

future clicks:
Would leak future information.

trend_direction:
Represents the outcome being predicted and therefore causes target leakage.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.